# Huấn Luyện & Đánh Giá So Sánh CKAN vs Matrix Factorization Trên Tập Dữ Liệu Book-Crossing (GPU Google Colab)

## Bối Cảnh Thực Nghiệm & Mục Tiêu

Trong các hệ thống gợi ý dựa trên Đồ thị Tri thức (Knowledge Graph - KG), hiệu quả vượt trội của KG thể hiện rõ rệt nhất khi làm việc với **dữ liệu cực kỳ thưa thớt (Extremely Sparse Data)** hoặc trong tình huống **Khởi đầu lạnh (Cold-start)**.

### So sánh tính chất dữ liệu: MovieLens-1M vs Book-Crossing

| Đặc tính | MovieLens-1M (Phim) | Book-Crossing (Sách) | Ý nghĩa thực nghiệm |
| :--- | :--- | :--- | :--- |
| Số User | ~6,040 | ~17,860 | Book-Crossing có quy mô người dùng lớn hơn gần 3 lần |
| Số Item | ~3,700 | ~14,967 | Kho sách phong phú gấp 4 lần kho phim |
| Số tương tác | ~1,000,000 | ~139,746 | Mật độ tương tác của Book-Crossing cực thấp |
| Tương tác TB / User | ~165 lượt / user | ~3.9 lượt / user | Người dùng đọc sách rất ít so với xem phim |
| Độ thưa thớt (Sparsity) | ~95.53% | **> 99.97%** | Không gian tương tác hầu như trống rỗng |
| Thực thể KG (Entities) | ~18,000 | **~77,903** | Đồ thị tri thức Satori về sách rất rộng lớn |
| Bộ ba KG (Triples) | ~20,000 | **~151,500** | Mạng lưới quan hệ Tác giả, Thể loại, Nhà XB đa dạng |

### Hiện tượng gì xảy ra với Matrix Factorization (MF)?
1. **Trên MovieLens (Dữ liệu dày - 165 lượt/user)**: MF chỉ học ma trận ID người dùng và ID phim. Do dữ liệu tương tác dày, MF ghi nhớ chính xác hành vi (memorization) và thiên lệch vào các phim phổ biến (popularity bias), dẫn đến chỉ số Recall@K trên tập test rất cao.
2. **Trên Book-Crossing (Dữ liệu thưa - 3.9 lượt/user)**: Mỗi người dùng chỉ đọc trung bình dưới 4 cuốn sách. Ma trận ID của MF rơi vào tình trạng thiếu dữ liệu trầm trọng (Sparsity bottleneck), không thể cân chỉnh vector biểu diễn người dùng và sách chưa từng gặp.

### Giải pháp từ CKAN (Collaborative Knowledge-aware Attentive Network):
- CKAN kết hợp tín hiệu cộng tác (Collaborative signals) và Đồ thị tri thức (KG Satori với 77,903 thực thể).
- Thông qua cơ chế **Lan truyền tri thức (KG Propagation L=2 bước nhảy)** và **Mạng Attention thích ứng**, CKAN kết nối người dùng với các cuốn sách chưa từng đọc qua các mối quan hệ ngữ nghĩa (Cùng Tác giả, Cùng Nhà xuất bản, Cùng Series tác phẩm, Cùng Thể loại).
- Kết quả: CKAN mang lại khả năng tổng quát hóa (generalization) và độ phủ gợi ý vượt trội trên dữ liệu thưa thớt.

In [ ]:
# ============================================================
# 1. KIỂM TRA MÔI TRƯỜNG PHẦN CỨNG GPU (GOOGLE COLAB)
# ============================================================
import torch
import subprocess
import os

print("--- THÔNG TIN PHẦN CỨNG ---")
gpu_available = torch.cuda.is_available()
if gpu_available:
    device = torch.device("cuda")
    props = torch.cuda.get_device_properties(0)
    print(f"  [OK] GPU Đang Hoạt Động : {props.name}")
    print(f"  [OK] Dung Lượng VRAM   : {props.total_memory / 1e9:.2f} GB")
    print(f"  [OK] Phiên Bản PyTorch : {torch.__version__}")
    print(f"  [OK] Phiên Bản CUDA    : {torch.version.cuda}")
else:
    device = torch.device("cpu")
    print("  [CẢNH BÁO] Không tìm thấy GPU. Hãy vào Runtime -> Change runtime type -> T4 GPU.")

print(f"Thiết bị tính toán: {device}")

In [ ]:
# ============================================================
# 2. KHAI BÁO THƯ VIỆN & CẤU HÌNH RANDOM SEED
# ============================================================
import sys
import time
import math
import collections
import urllib.request
import numpy as np
import pandas as pd
from collections import defaultdict
from sklearn.metrics import roc_auc_score, f1_score, accuracy_score
from tqdm import tqdm
import matplotlib.pyplot as plt
import seaborn as sns

# Thiết lập Random Seed đồng nhất để tái lập kết quả
SEED = 2026
np.random.seed(SEED)
torch.manual_seed(SEED)
if gpu_available:
    torch.cuda.manual_seed(SEED)
    torch.cuda.manual_seed_all(SEED)

print("Khai báo thư viện và thiết lập seed hoàn tất.")

In [ ]:
# ============================================================
# 3. THIẾT LẬP SIÊU THAM SỐ (HYPERPARAMETERS CHO BOOK-CROSSING)
# ============================================================
DATASET = "book"

# Cấu hình chuẩn theo công bố bài báo khoa học CKAN (SIGIR 2020) cho Book-Crossing:
DIM           = 64       # Số chiều vector biểu diễn Embedding
N_LAYER       = 2        # Số bước lan truyền tri thức (L=2 tối ưu cho Book-Crossing)
UTSS          = 16       # Kích thước tập bộ ba user (User Triple Set Size)
ITSS          = 64       # Kích thước tập bộ ba item (Item Triple Set Size)
AGG           = "concat" # Cơ chế gộp embedding: 'concat' | 'sum' | 'pool'
BATCH_SIZE    = 1024     # Kích thước mini-batch
N_EPOCH       = 15       # Số epoch huấn luyện
LEARNING_RATE = 0.002    # Tốc độ học của Adam Optimizer
L2_WEIGHT     = 1e-5     # Hệ số điều chuẩn L2 Weight Decay

DATA_DIR = f"./data/{DATASET}"
os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs("./models", exist_ok=True)

print(f"Dataset               : {DATASET}")
print(f"Embedding Dimension   : {DIM}")
print(f"KG Propagation Layers : {N_LAYER}")
print(f"User Triple Set Size  : {UTSS}")
print(f"Item Triple Set Size  : {ITSS}")
print(f"Aggregator            : {AGG}")
print(f"Batch Size            : {BATCH_SIZE}")
print(f"Epochs                : {N_EPOCH}")
print(f"Learning Rate         : {LEARNING_RATE}")

## 4. Chuẩn Bị Dữ Liệu Book-Crossing & KG Satori
Quy trình chuẩn bị dữ liệu hỗ trợ cơ chế kép (Resilient Dual-Mode):
1. **Chế độ nhanh (Fast-track)**: Tải trực tiếp các file ma trận đã tiền xử lý chuẩn (`ratings_final.npy` và `kg_final.npy`) từ kho lưu trữ GitHub của đồ án.
2. **Chế độ tự động xử lý (Auto-Fallback)**: Nếu liên kết file `.npy` chưa sẵn sàng, hệ thống tự động tải file gốc `BX-Book-Ratings.csv` và `kg.txt`, thực hiện chuyển đổi và negative sampling trong ~30 giây.

In [ ]:
# ============================================================
# 4. TẢI VÀ CHUẨN BỊ DỮ LIỆU BOOK-CROSSING
# ============================================================
REPO_RAW_URL = "https://raw.githubusercontent.com/Chinh-de/dss_ckan/main/backend/data/book"
RIPPLE_BOOK_URL = "https://raw.githubusercontent.com/hwwang55/RippleNet/master/data/book"
CKAN_BOOK_URL = "https://raw.githubusercontent.com/weberrr/CKAN/master/data/book"

def download_file(url, dest):
    if os.path.exists(dest) and os.path.getsize(dest) > 1000:
        print(f"  [Đã tồn tại] {dest} ({os.path.getsize(dest)/1e6:.2f} MB)")
        return True
    try:
        print(f"  [Đang tải] {url} -> {dest}")
        urllib.request.urlretrieve(url, dest)
        print(f"  [Hoàn tất] {dest} ({os.path.getsize(dest)/1e6:.2f} MB)")
        return True
    except Exception as e:
        print(f"  [Lỗi tải {url}]: {e}")
        return False

# Bước 1: Ưu tiên tải ratings_final.npy và kg_final.npy
has_npy = False
if download_file(f"{REPO_RAW_URL}/ratings_final.npy", f"{DATA_DIR}/ratings_final.npy") and    download_file(f"{REPO_RAW_URL}/kg_final.npy", f"{DATA_DIR}/kg_final.npy"):
    has_npy = True

# Bước 2: Tải các file ánh xạ entity ID
download_file(f"{CKAN_BOOK_URL}/item_index2entity_id.txt", f"{DATA_DIR}/item_index2entity_id.txt")

# Bước 3: Nếu chưa có file .npy, tự động tiền xử lý từ file gốc
if not has_npy:
    print("")
    print("File .npy chưa có sẵn, tiến hành tải file gốc và tiền xử lý tự động...")
    download_file(f"{CKAN_BOOK_URL}/kg.txt", f"{DATA_DIR}/kg.txt")
    download_file(f"{RIPPLE_BOOK_URL}/BX-Book-Ratings.csv", f"{DATA_DIR}/BX-Book-Ratings.csv")

    print("Bắt đầu tiền xử lý dữ liệu Book-Crossing...")
    t0 = time.time()
    
    # 1. Đọc ánh xạ item
    item_index_old2new = {}
    entity_id2index = {}
    with open(f"{DATA_DIR}/item_index2entity_id.txt", "r", encoding="utf-8") as f:
        for idx, line in enumerate(f):
            it, ent = line.strip().split("\t")
            item_index_old2new[it] = idx
            entity_id2index[ent] = idx

    # 2. Xử lý rating và negative sampling
    item_set = set(item_index_old2new.values())
    user_pos_ratings = defaultdict(set)
    user_neg_ratings = defaultdict(set)
    with open(f"{DATA_DIR}/BX-Book-Ratings.csv", "r", encoding="utf-8") as f:
        lines = f.readlines()[1:]
        for line in lines:
            parts = [p.strip('"') for p in line.strip().split(";")]
            if len(parts) < 3:
                continue
            u_old, it_old, score_str = parts[0], parts[1], parts[2]
            if it_old not in item_index_old2new:
                continue
            it_new = item_index_old2new[it_old]
            try:
                score = float(score_str)
            except ValueError:
                continue
            if score >= 0:
                user_pos_ratings[u_old].add(it_new)
            else:
                user_neg_ratings[u_old].add(it_new)

    ratings_list = []
    user_index_old2new = {}
    u_cnt = 0
    np.random.seed(555)
    for u_old, pos_items in user_pos_ratings.items():
        if u_old not in user_index_old2new:
            user_index_old2new[u_old] = u_cnt
            u_cnt += 1
        u_new = user_index_old2new[u_old]
        for it in pos_items:
            ratings_list.append([u_new, it, 1])
        unwatched = list(item_set - pos_items - user_neg_ratings[u_old])
        if len(unwatched) >= len(pos_items):
            neg_items = np.random.choice(unwatched, size=len(pos_items), replace=False)
            for it in neg_items:
                ratings_list.append([u_new, it, 0])

    rating_np = np.array(ratings_list, dtype=np.int32)
    np.save(f"{DATA_DIR}/ratings_final.npy", rating_np)

    # 3. Xử lý Knowledge Graph
    ent_cnt = len(entity_id2index)
    rel_id2index = {}
    rel_cnt = 0
    kg_triples = []
    with open(f"{DATA_DIR}/kg.txt", "r", encoding="utf-8") as f:
        for line in f:
            h_old, r_old, t_old = line.strip().split("\t")
            if h_old not in entity_id2index:
                entity_id2index[h_old] = ent_cnt
                ent_cnt += 1
            h = entity_id2index[h_old]
            if t_old not in entity_id2index:
                entity_id2index[t_old] = ent_cnt
                ent_cnt += 1
            t = entity_id2index[t_old]
            if r_old not in rel_id2index:
                rel_id2index[r_old] = rel_cnt
                rel_cnt += 1
            r = rel_id2index[r_old]
            kg_triples.append([h, r, t])

    kg_np = np.array(kg_triples, dtype=np.int32)
    np.save(f"{DATA_DIR}/kg_final.npy", kg_np)
    print(f"Tiền xử lý hoàn tất trong {time.time() - t0:.2f}s!")

# Nạp ma trận dữ liệu
rating_np = np.load(f"{DATA_DIR}/ratings_final.npy")
kg_np = np.load(f"{DATA_DIR}/kg_final.npy")

n_user = int(np.max(rating_np[:, 0])) + 1
n_item = int(np.max(rating_np[:, 1])) + 1
n_entity = int(max(np.max(kg_np[:, 0]), np.max(kg_np[:, 2]))) + 1
n_relation = int(np.max(kg_np[:, 1])) + 1

pos_ratings_count = int(np.sum(rating_np[:, 2] == 1))
total_possible_pairs = n_user * n_item
sparsity_pct = (1.0 - (pos_ratings_count / total_possible_pairs)) * 100.0

print("")
print(f"--- BÁO CÁO THỐNG KÊ TẬP DỮ LIỆU BOOK-CROSSING ---")
print(f"  - Số lượng Người dùng (Users)      : {n_user:,}")
print(f"  - Số lượng Đầu sách (Books/Items)  : {n_item:,}")
print(f"  - Tổng số tương tác mẫu (Rating)   : {len(rating_np):,} (Dương: {pos_ratings_count:,})")
print(f"  - Tương tác dương TB / Người dùng  : {pos_ratings_count / n_user:.2f} sách/user")
print(f"  - Độ thưa thớt không gian (Sparsity): {sparsity_pct:.4f}%")
print(f"  - Số lượng Thực thể (KG Entities)  : {n_entity:,}")
print(f"  - Số lượng Quan hệ (KG Relations)  : {n_relation:,}")
print(f"  - Tổng số bộ ba tri thức (Triples) : {len(kg_np):,}")

## 5. Phân Chia Dữ Liệu (6:2:2) & Lan Truyền Tri Thức (KG Propagation)

- Tập dữ liệu được phân chia theo tỷ lệ chuẩn `60% Train : 20% Validation : 20% Test`.
- Mạng lan truyền tri thức được xây dựng qua `N_LAYER = 2` bước nhảy:
  - **Tầng 0 (User History)**: Lấy mẫu các cuốn sách người dùng đã đọc trong tập Train làm điểm tựa ban đầu.
  - **Tầng 1 (1-hop Triples)**: Tìm kiếm các bộ ba tri thức `(h, r, t)` trực tiếp gắn liền với cuốn sách (Tác giả, Năm phát hành, Nhà XB).
  - **Tầng 2 (2-hop Triples)**: Mở rộng sang các thực thể kế tiếp (ví dụ: Các tác phẩm khác của cùng tác giả, các giải thưởng văn học, thể loại phái sinh).

In [ ]:
# ============================================================
# 5. PHÂN CHIA DỮ LIỆU & LAN TRUYỀN TRI THỨC (KG PROPAGATION)
# ============================================================
np.random.seed(SEED)

def dataset_split(rating_np):
    n_rows = rating_np.shape[0]
    idx = np.random.permutation(n_rows)
    eval_end = int(n_rows * 0.6)
    test_end = int(n_rows * 0.8)

    train_data = rating_np[idx[:eval_end]]
    eval_data  = rating_np[idx[eval_end:test_end]]
    test_data  = rating_np[idx[test_end:]]

    user_seed = defaultdict(list)
    item_seed = defaultdict(list)
    for u, i, r in train_data:
        if r == 1:
            user_seed[u].append(i)
            item_seed[i].append(u)
    return train_data, eval_data, test_data, user_seed, item_seed

def construct_kg(kg_np):
    kg = defaultdict(list)
    for h, r, t in kg_np:
        kg[h].append((h, r, t))
    return kg

def kg_propagation(kg, seed_dict, set_size, n_layer):
    triple_sets = {}
    for obj, seeds in seed_dict.items():
        layers = []
        if len(seeds) == 0:
            layers.append(([0] * set_size, [0] * set_size, [0] * set_size))
        else:
            idx = np.random.choice(len(seeds), size=set_size, replace=(len(seeds) < set_size))
            layers.append(([seeds[i] for i in idx], [0] * set_size, [0] * set_size))

        for l in range(n_layer):
            h_prev = layers[-1][0] if l == 0 else layers[-1][2]
            h_list, r_list, t_list = [], [], []
            for h in h_prev:
                triples = kg.get(h, [])
                if len(triples) == 0:
                    continue
                chosen_idx = np.random.choice(len(triples))
                chosen = triples[chosen_idx]
                h_list.append(chosen[0])
                r_list.append(chosen[1])
                t_list.append(chosen[2])

            if len(h_list) == 0:
                layers.append(layers[-1] if l > 0 else ([0] * set_size, [0] * set_size, [0] * set_size))
            else:
                idx = np.random.choice(len(h_list), size=set_size, replace=(len(h_list) < set_size))
                layers.append(([h_list[i] for i in idx], [r_list[i] for i in idx], [t_list[i] for i in idx]))
        triple_sets[obj] = layers
    return triple_sets

print("Đang phân chia dữ liệu và xây dựng mạng lan truyền tri thức...")
train_data, eval_data, test_data, user_seed, item_seed = dataset_split(rating_np)
kg = construct_kg(kg_np)
user_triple_set = kg_propagation(kg, user_seed, UTSS, N_LAYER)
item_triple_set = kg_propagation(kg, item_seed, ITSS, N_LAYER)

print(f"  - Tập Huấn luyện (Train): {train_data.shape[0]:,} mẫu")
print(f"  - Tập Kiểm định (Eval)  : {eval_data.shape[0]:,} mẫu")
print(f"  - Tập Thử nghiệm (Test) : {test_data.shape[0]:,} mẫu")
print(f"  - Người dùng có tri thức: {len(user_triple_set):,}")
print(f"  - Đầu sách có tri thức  : {len(item_triple_set):,}")
print("Phân chia và lan truyền tri thức hoàn tất.")

## 6. Định Nghĩa Cấu Trúc 2 Mô Hình
Định nghĩa kiến trúc hoàn chỉnh cho 2 trường phái giải thuật:
1. **Matrix Factorization (MF Baseline)**: Phương pháp Lọc cộng tác truyền thống, chỉ dùng ma trận ID User và Item cùng hệ số bias, không có thông tin ngữ nghĩa.
2. **CKAN (With Knowledge Graph)**: Mạng nơ-ron tích hợp tri thức với cơ chế Attention nhiều bước nhảy (Multi-hop Knowledge Attention) và cơ chế gộp vector Concatenation.

In [ ]:
# ============================================================
# 6. ĐỊNH NGHĨA KIẾN TRÚC MÔ HÌNH (MF vs CKAN)
# ============================================================
import torch.nn as nn
import torch.nn.functional as F

# --- 6A. BASELINE: BIASED MATRIX FACTORIZATION (MF - NO KG) ---
class MatrixFactorization(nn.Module):
    """
    Biased Matrix Factorization: Chỉ học từ ma trận tương tác User-Item ID.
    """
    def __init__(self, n_user, n_item, dim):
        super().__init__()
        self.user_emb = nn.Embedding(n_user, dim)
        self.item_emb = nn.Embedding(n_item, dim)
        self.user_bias = nn.Embedding(n_user, 1)
        self.item_bias = nn.Embedding(n_item, 1)

        nn.init.xavier_uniform_(self.user_emb.weight)
        nn.init.xavier_uniform_(self.item_emb.weight)
        nn.init.zeros_(self.user_bias.weight)
        nn.init.zeros_(self.item_bias.weight)

    def forward(self, users, items):
        u = self.user_emb(users)
        v = self.item_emb(items)
        dot = (u * v).sum(dim=-1, keepdim=True)
        logits = dot + self.user_bias(users) + self.item_bias(items)
        return torch.sigmoid(logits.squeeze(-1))

    def get_all_scores(self, user_indices):
        u = self.user_emb(user_indices)
        v = self.item_emb.weight
        u_b = self.user_bias(user_indices)
        v_b = self.item_bias.weight.T
        logits = torch.matmul(u, v.T) + u_b + v_b
        return torch.sigmoid(logits)


# --- 6B. ĐỀ XUẤT: CKAN (WITH KNOWLEDGE GRAPH) ---
class CKAN(nn.Module):
    """
    Collaborative Knowledge-aware Attentive Network (CKAN).
    Sử dụng mạng Attention 3 tầng để tổng hợp ngữ nghĩa từ Đồ thị Tri thức.
    """
    def __init__(self, n_entity, n_relation, dim, n_layer=2, agg="concat"):
        super().__init__()
        self.n_entity = n_entity
        self.n_relation = n_relation
        self.dim = dim
        self.n_layer = n_layer
        self.agg = agg

        self.entity_emb = nn.Embedding(n_entity, dim)
        self.relation_emb = nn.Embedding(n_relation, dim)

        self.attention = nn.Sequential(
            nn.Linear(dim * 2, dim, bias=False),
            nn.ReLU(),
            nn.Linear(dim, dim, bias=False),
            nn.ReLU(),
            nn.Linear(dim, 1, bias=False),
            nn.Sigmoid()
        )
        self._init_weight()

    def _init_weight(self):
        nn.init.xavier_uniform_(self.entity_emb.weight)
        nn.init.xavier_uniform_(self.relation_emb.weight)
        for m in self.attention:
            if isinstance(m, nn.Linear):
                nn.init.xavier_uniform_(m.weight)

    def _knowledge_attention(self, h_emb, r_emb, t_emb):
        # [batch_size, triple_set_size]
        att_weights = self.attention(torch.cat((h_emb, r_emb), dim=-1)).squeeze(-1)
        att_norm = F.softmax(att_weights, dim=-1)
        emb = torch.mul(att_norm.unsqueeze(-1), t_emb).sum(dim=1)
        return emb

    def get_user_embeddings(self, user_triple):
        user_embs = [self.entity_emb(user_triple[0][0]).mean(dim=1)]
        for l in range(self.n_layer):
            h = self.entity_emb(user_triple[0][l])
            r = self.relation_emb(user_triple[1][l])
            t = self.entity_emb(user_triple[2][l])
            user_embs.append(self._knowledge_attention(h, r, t))

        e_u = user_embs[0]
        if self.agg == "concat":
            for i in range(1, len(user_embs)):
                e_u = torch.cat((user_embs[i], e_u), dim=-1)
        elif self.agg == "sum":
            for i in range(1, len(user_embs)):
                e_u = e_u + user_embs[i]
        return e_u

    def get_item_embeddings(self, items, item_triple):
        item_embs = [self.entity_emb(items)]
        for l in range(self.n_layer):
            h = self.entity_emb(item_triple[0][l])
            r = self.relation_emb(item_triple[1][l])
            t = self.entity_emb(item_triple[2][l])
            item_embs.append(self._knowledge_attention(h, r, t))

        e_v = item_embs[0]
        if self.agg == "concat":
            for i in range(1, len(item_embs)):
                e_v = torch.cat((item_embs[i], e_v), dim=-1)
        elif self.agg == "sum":
            for i in range(1, len(item_embs)):
                e_v = e_v + item_embs[i]
        return e_v

    def forward(self, items, user_triple, item_triple):
        e_u = self.get_user_embeddings(user_triple)
        e_v = self.get_item_embeddings(items, item_triple)
        return torch.sigmoid((e_v * e_u).sum(dim=-1))

print("Đã khởi tạo xong cấu trúc 2 mô hình (MF và CKAN).")

## 7. Huấn Luyện Mô Hình Baseline: Matrix Factorization (MF)
Thực hiện huấn luyện mô hình cơ sở không có Đồ thị Tri thức để ghi nhận khả năng học trên dữ liệu cực thưa.

In [ ]:
# ============================================================
# 7. HUẤN LUYỆN MATRIX FACTORIZATION (MF BASELINE)
# ============================================================
def evaluate_mf(model, data, b_size=BATCH_SIZE):
    model.eval()
    aucs, f1s = [], []
    with torch.no_grad():
        for start in range(0, data.shape[0], b_size):
            end = min(start + b_size, data.shape[0])
            batch = data[start:end]
            u = torch.LongTensor(batch[:, 0]).to(device)
            v = torch.LongTensor(batch[:, 1]).to(device)
            scores = model(u, v).cpu().numpy()
            labels = batch[:, 2]
            if len(np.unique(labels)) > 1:
                aucs.append(roc_auc_score(labels, scores))
                f1s.append(f1_score(labels, (scores >= 0.5).astype(int), zero_division=0))
    model.train()
    return float(np.mean(aucs)), float(np.mean(f1s))

mf_model = MatrixFactorization(n_user, n_item, DIM).to(device)
mf_optimizer = torch.optim.Adam(mf_model.parameters(), lr=LEARNING_RATE, weight_decay=L2_WEIGHT)
loss_fn = nn.BCELoss()

print(f"--- BẮT ĐẦU HUẤN LUYỆN MATRIX FACTORIZATION ({N_EPOCH} EPOCHS) ---")
mf_history = {"epoch": [], "train_loss": [], "eval_auc": [], "eval_f1": []}
best_mf_auc = 0.0

for epoch in range(1, N_EPOCH + 1):
    t_start = time.time()
    np.random.shuffle(train_data)
    total_loss, steps = 0.0, 0
    mf_model.train()

    for start in range(0, train_data.shape[0], BATCH_SIZE):
        end = min(start + BATCH_SIZE, train_data.shape[0])
        batch = train_data[start:end]

        u = torch.LongTensor(batch[:, 0]).to(device)
        v = torch.LongTensor(batch[:, 1]).to(device)
        labels = torch.FloatTensor(batch[:, 2]).to(device)

        scores = mf_model(u, v)
        loss = loss_fn(scores, labels)

        mf_optimizer.zero_grad()
        loss.backward()
        mf_optimizer.step()

        total_loss += loss.item()
        steps += 1

    train_loss = total_loss / steps
    eval_auc, eval_f1 = evaluate_mf(mf_model, eval_data)
    mf_history["epoch"].append(epoch)
    mf_history["train_loss"].append(train_loss)
    mf_history["eval_auc"].append(eval_auc)
    mf_history["eval_f1"].append(eval_f1)

    if eval_auc > best_mf_auc:
        best_mf_auc = eval_auc
        torch.save(mf_model.state_dict(), "./models/mf_book_best.pt")

    duration = time.time() - t_start
    print(f"  Epoch {epoch:2d}/{N_EPOCH} [{duration:4.1f}s] - Train Loss: {train_loss:.4f} | Eval AUC: {eval_auc:.4f} | Eval F1: {eval_f1:.4f}")

# Nạp checkpoint tốt nhất và kiểm tra trên Test
mf_model.load_state_dict(torch.load("./models/mf_book_best.pt"))
mf_test_auc, mf_test_f1 = evaluate_mf(mf_model, test_data)
print("")
print(f"[KẾT QUẢ MF TEST] Test AUC: {mf_test_auc:.4f} | Test F1: {mf_test_f1:.4f}")

## 8. Huấn Luyện Mô Hình Đề Xuất: CKAN (With Knowledge Graph)
Huấn luyện mô hình CKAN với 2 tầng lan truyền tri thức (L=2) và tự động ghi nhận checkpoint tốt nhất.

In [ ]:
# ============================================================
# 8. HUẤN LUYỆN CKAN (WITH KNOWLEDGE GRAPH)
# ============================================================
default_user_triples = list(user_triple_set.values())[0]
default_item_triples = list(item_triple_set.values())[0]

def to_triple_tensor(objs, triple_set, default_triples, n_layer, dev):
    h, r, t = [], [], []
    for l in range(n_layer):
        h.append(torch.LongTensor([triple_set.get(o, default_triples)[l][0] for o in objs]).to(dev))
        r.append(torch.LongTensor([triple_set.get(o, default_triples)[l][1] for o in objs]).to(dev))
        t.append(torch.LongTensor([triple_set.get(o, default_triples)[l][2] for o in objs]).to(dev))
    return [h, r, t]

def evaluate_ckan(model, data, b_size=BATCH_SIZE):
    model.eval()
    aucs, f1s = [], []
    with torch.no_grad():
        for start in range(0, data.shape[0], b_size):
            end = min(start + b_size, data.shape[0])
            batch = data[start:end]
            items = torch.LongTensor(batch[:, 1]).to(device)
            u_tr = to_triple_tensor(batch[:, 0].tolist(), user_triple_set, default_user_triples, N_LAYER, device)
            i_tr = to_triple_tensor(batch[:, 1].tolist(), item_triple_set, default_item_triples, N_LAYER, device)
            scores = model(items, u_tr, i_tr).cpu().numpy()
            labels = batch[:, 2]
            if len(np.unique(labels)) > 1:
                aucs.append(roc_auc_score(labels, scores))
                f1s.append(f1_score(labels, (scores >= 0.5).astype(int), zero_division=0))
    model.train()
    return float(np.mean(aucs)), float(np.mean(f1s))

ckan_model = CKAN(n_entity, n_relation, DIM, n_layer=N_LAYER, agg=AGG).to(device)
ckan_optimizer = torch.optim.Adam(ckan_model.parameters(), lr=LEARNING_RATE, weight_decay=L2_WEIGHT)

print(f"--- BẮT ĐẦU HUẤN LUYỆN CKAN ({N_EPOCH} EPOCHS) ---")
ckan_history = {"epoch": [], "train_loss": [], "eval_auc": [], "eval_f1": []}
best_ckan_auc = 0.0

for epoch in range(1, N_EPOCH + 1):
    t_start = time.time()
    np.random.shuffle(train_data)
    total_loss, steps = 0.0, 0
    ckan_model.train()

    for start in range(0, train_data.shape[0], BATCH_SIZE):
        end = min(start + BATCH_SIZE, train_data.shape[0])
        batch = train_data[start:end]

        items = torch.LongTensor(batch[:, 1]).to(device)
        u_tr = to_triple_tensor(batch[:, 0].tolist(), user_triple_set, default_user_triples, N_LAYER, device)
        i_tr = to_triple_tensor(batch[:, 1].tolist(), item_triple_set, default_item_triples, N_LAYER, device)
        labels = torch.FloatTensor(batch[:, 2]).to(device)

        scores = ckan_model(items, u_tr, i_tr)
        loss = loss_fn(scores, labels)

        ckan_optimizer.zero_grad()
        loss.backward()
        ckan_optimizer.step()

        total_loss += loss.item()
        steps += 1

    train_loss = total_loss / steps
    eval_auc, eval_f1 = evaluate_ckan(ckan_model, eval_data)
    ckan_history["epoch"].append(epoch)
    ckan_history["train_loss"].append(train_loss)
    ckan_history["eval_auc"].append(eval_auc)
    ckan_history["eval_f1"].append(eval_f1)

    if eval_auc > best_ckan_auc:
        best_ckan_auc = eval_auc
        torch.save(ckan_model.state_dict(), "./models/ckan_book_best.pt")

    duration = time.time() - t_start
    print(f"  Epoch {epoch:2d}/{N_EPOCH} [{duration:4.1f}s] - Train Loss: {train_loss:.4f} | Eval AUC: {eval_auc:.4f} | Eval F1: {eval_f1:.4f}")

# Nạp checkpoint tốt nhất và kiểm tra trên Test
ckan_model.load_state_dict(torch.load("./models/ckan_book_best.pt"))
ckan_test_auc, ckan_test_f1 = evaluate_ckan(ckan_model, test_data)
print("")
print(f"[KẾT QUẢ CKAN TEST] Test AUC: {ckan_test_auc:.4f} | Test F1: {ckan_test_f1:.4f}")

## 9. Đánh Giá Top-K Recommendation (Vectorized GPU All-Ranking Protocol)

- Giao thức **All-Ranking**: Chấm điểm toàn bộ các cuốn sách trong kho ({n_item:,} sách) cho từng người dùng trong tập test, loại trừ các cuốn đã đọc trong tập Train.
- Tối ưu hóa **GPU Vectorization**:
  - Trích xuất toàn bộ vector biểu diễn người dùng $E_U \in \mathbb{R}^{N_{eval} \times D}$ và vector đầu sách $E_V \in \mathbb{R}^{N_{items} \times D}$.
  - Thực hiện nhân ma trận song song $E_U \times E_V^T$ trên GPU theo từng cụm (batch) trong ~1 giây, thay thế hoàn toàn vòng lặp tuần tự cũ.
- Thang đo đánh giá:
  - **Recall@K**: Tỷ lệ các cuốn sách người dùng thực sự đọc trong tập test xuất hiện trong top K gợi ý.
  - **NDCG@K**: Đánh giá vị trí xếp hạng (sách đúng được xếp càng cao thì điểm càng cao).

In [ ]:
# ============================================================
# 9. ĐÁNH GIÁ TOP-K RECOMMENDATION (GPU VECTORIZED)
# ============================================================
def dcg_at_k(r, k):
    r = np.asarray(r, dtype=float)[:k]
    if r.size:
        return np.sum(r / np.log2(np.arange(2, r.size + 2)))
    return 0.0

def ndcg_at_k(r, k, ground_truth_count):
    dcg = dcg_at_k(r, k)
    ideal_r = [1] * min(k, ground_truth_count)
    idcg = dcg_at_k(ideal_r, k)
    return dcg / idcg if idcg > 0 else 0.0

def topk_eval_all(mf_model, ckan_model, train_data, test_data, user_triple_set, item_triple_set, n_layer, n_item, device, k_list=[5, 10, 20]):
    mf_model.eval()
    ckan_model.eval()

    user_train_pos = defaultdict(set)
    for u, i, r in train_data:
        if r == 1:
            user_train_pos[u].add(i)

    user_test_pos = defaultdict(set)
    for u, i, r in test_data:
        if r == 1:
            user_test_pos[u].add(i)

    # Đánh giá trên 100% người dùng có dữ liệu test
    eval_users = sorted(list(set(user_train_pos.keys()) & set(user_test_pos.keys())))
    all_items = np.arange(n_item)

    print(f"Đang tính ma trận Top-K ranking cho {n_item:,} đầu sách và {len(eval_users):,} người dùng test...")

    with torch.no_grad():
        # --- 1. ĐÁNH GIÁ MF TRÊN GPU ---
        mf_u_tens = torch.LongTensor(eval_users).to(device)
        mf_all_scores = mf_model.get_all_scores(mf_u_tens)

        # --- 2. ĐÁNH GIÁ CKAN TRÊN GPU ---
        item_tensor = torch.LongTensor(all_items).to(device)
        all_i_tr = to_triple_tensor(all_items.tolist(), item_triple_set, default_item_triples, n_layer, device)
        E_V = ckan_model.get_item_embeddings(item_tensor, all_i_tr)

        all_u_tr = to_triple_tensor(eval_users, user_triple_set, default_user_triples, n_layer, device)
        E_U = ckan_model.get_user_embeddings(all_u_tr)
        ckan_all_scores = torch.sigmoid(torch.matmul(E_U, E_V.T))

        # Loại trừ các sách đã đọc trong tập Train
        for idx, u in enumerate(eval_users):
            seen = list(user_train_pos[u])
            if len(seen) > 0:
                mf_all_scores[idx, seen] = -1e9
                ckan_all_scores[idx, seen] = -1e9

        mf_top_indices = torch.topk(mf_all_scores, k=max(k_list), dim=-1).indices.cpu().numpy()
        ckan_top_indices = torch.topk(ckan_all_scores, k=max(k_list), dim=-1).indices.cpu().numpy()

    mf_recalls, mf_ndcgs = {k: [] for k in k_list}, {k: [] for k in k_list}
    ckan_recalls, ckan_ndcgs = {k: [] for k in k_list}, {k: [] for k in k_list}

    print(f"Đang tổng hợp thang đo trên toàn bộ {len(eval_users):,} người dùng...")
    for idx, u in enumerate(eval_users):
        ground_truth = user_test_pos[u]
        mf_user_top = mf_top_indices[idx]
        ckan_user_top = ckan_top_indices[idx]

        for k in k_list:
            # MF
            mf_hits = [1 if item in ground_truth else 0 for item in mf_user_top[:k]]
            mf_recalls[k].append(sum(mf_hits) / len(ground_truth))
            mf_ndcgs[k].append(ndcg_at_k(mf_hits, k, len(ground_truth)))

            # CKAN
            ckan_hits = [1 if item in ground_truth else 0 for item in ckan_user_top[:k]]
            ckan_recalls[k].append(sum(ckan_hits) / len(ground_truth))
            ckan_ndcgs[k].append(ndcg_at_k(ckan_hits, k, len(ground_truth)))

    return {
        "MF": {
            "Recall": {k: float(np.mean(mf_recalls[k])) for k in k_list},
            "NDCG": {k: float(np.mean(mf_ndcgs[k])) for k in k_list}
        },
        "CKAN": {
            "Recall": {k: float(np.mean(ckan_recalls[k])) for k in k_list},
            "NDCG": {k: float(np.mean(ckan_ndcgs[k])) for k in k_list}
        }
    }

topk_cmp = topk_eval_all(mf_model, ckan_model, train_data, test_data, user_triple_set, item_triple_set, N_LAYER, n_item, device, k_list=[5, 10, 20])

print("-" * 80)
print(f"{'K':<5} | {'MF Recall@K':<15} {'CKAN Recall@K':<15} | {'MF NDCG@K':<15} {'CKAN NDCG@K':<15}")
print("-" * 80)
for k in [5, 10, 20]:
    mf_r = topk_cmp['MF']['Recall'][k]
    ckan_r = topk_cmp['CKAN']['Recall'][k]
    mf_n = topk_cmp['MF']['NDCG'][k]
    ckan_n = topk_cmp['CKAN']['NDCG'][k]
    print(f"{k:<5} | {mf_r:<15.4f} {ckan_r:<15.4f} | {mf_n:<15.4f} {ckan_n:<15.4f}")
print("-" * 80)

## 10. Trực Quan Hóa So Sánh Toàn Diện (CTR Prediction & Top-K Recommendation)
Vẽ 4 biểu đồ trực quan phân tích sự vượt trội của CKAN trên dữ liệu cực thưa.

In [ ]:
# ============================================================
# 10. TRỰC QUAN HÓA SO SÁNH HIỆU NĂNG
# ============================================================
sns.set_theme(style="whitegrid")
fig, axes = plt.subplots(2, 2, figsize=(15, 11))

epochs = mf_history["epoch"]

# 1. So sánh Test ROC-AUC và F1-Score
bars = axes[0, 0].bar(
    ["MF (No KG)", "CKAN (With KG)"],
    [mf_test_auc, ckan_test_auc],
    color=["#94a3b8", "#3b82f6"],
    width=0.45
)
axes[0, 0].set_title("Test ROC-AUC Comparison (Book-Crossing)", fontsize=13, fontweight="bold")
axes[0, 0].set_ylim(0.5, 0.9)
for bar in bars:
    yval = bar.get_height()
    axes[0, 0].text(bar.get_x() + bar.get_width() / 2.0, yval + 0.008, f"{yval:.4f}", ha="center", va="bottom", fontweight="bold")

# 2. So sánh Recall@K
k_vals = [5, 10, 20]
mf_r_vals = [topk_cmp["MF"]["Recall"][k] for k in k_vals]
ckan_r_vals = [topk_cmp["CKAN"]["Recall"][k] for k in k_vals]

axes[0, 1].plot(k_vals, mf_r_vals, marker="o", linewidth=2.5, linestyle="--", color="#64748b", label="MF (Baseline)")
axes[0, 1].plot(k_vals, ckan_r_vals, marker="s", linewidth=2.5, color="#2563eb", label="CKAN (With KG)")
axes[0, 1].set_title("Top-K Recall@K (All-Ranking)", fontsize=13, fontweight="bold")
axes[0, 1].set_xlabel("K (Số lượng gợi ý)")
axes[0, 1].set_ylabel("Recall@K")
axes[0, 1].set_xticks(k_vals)
axes[0, 1].legend(loc="upper left")

# 3. So sánh NDCG@K
mf_n_vals = [topk_cmp["MF"]["NDCG"][k] for k in k_vals]
ckan_n_vals = [topk_cmp["CKAN"]["NDCG"][k] for k in k_vals]

axes[1, 0].plot(k_vals, mf_n_vals, marker="o", linewidth=2.5, linestyle="--", color="#64748b", label="MF (Baseline)")
axes[1, 0].plot(k_vals, ckan_n_vals, marker="s", linewidth=2.5, color="#059669", label="CKAN (With KG)")
axes[1, 0].set_title("Top-K NDCG@K Ranking Quality", fontsize=13, fontweight="bold")
axes[1, 0].set_xlabel("K (Số lượng gợi ý)")
axes[1, 0].set_ylabel("NDCG@K")
axes[1, 0].set_xticks(k_vals)
axes[1, 0].legend(loc="upper left")

# 4. Đường cong Loss khi huấn luyện
axes[1, 1].plot(epochs, mf_history["train_loss"], label="MF Train Loss", color="#64748b", linestyle="--")
axes[1, 1].plot(epochs, ckan_history["train_loss"], label="CKAN Train Loss", color="#dc2626")
axes[1, 1].set_title("Learning Curve (Train BCE Loss)", fontsize=13, fontweight="bold")
axes[1, 1].set_xlabel("Epoch")
axes[1, 1].set_ylabel("Loss")
axes[1, 1].legend()

plt.tight_layout()
plt.savefig("./models/comparison_book_crossing.png", dpi=300)
plt.show()

# --- TỔNG KẾT BẢNG SỐ LIỆU ---
print("")
print("=" * 75)
print("--- BẢNG TỔNG KẾT SO SÁNH HIỆU QUẢ TRÊN DỮ LIỆU BOOK-CROSSING ---")
print("=" * 75)
print(f"1. DỰ ĐOÁN CTR (CLICK-THROUGH RATE):")
print(f"   MF   Test AUC: {mf_test_auc:.4f} | F1: {mf_test_f1:.4f}")
print(f"   CKAN Test AUC: {ckan_test_auc:.4f} | F1: {ckan_test_f1:.4f}")
print("")
print(f"2. GỢI Ý TOP-K RECOMMENDATION (ALL-RANKING):")
for k in [5, 10, 20]:
    r_diff = ((ckan_r_vals[k_vals.index(k)] - mf_r_vals[k_vals.index(k)]) / max(mf_r_vals[k_vals.index(k)], 1e-6)) * 100
    n_diff = ((ckan_n_vals[k_vals.index(k)] - mf_n_vals[k_vals.index(k)]) / max(mf_n_vals[k_vals.index(k)], 1e-6)) * 100
    print(f"   K={k:2d}: Recall@{k}: MF={mf_r_vals[k_vals.index(k)]:.4f} vs CKAN={ckan_r_vals[k_vals.index(k)]:.4f} (CKAN thay đổi: {r_diff:+.2f}%)")
    print(f"         NDCG@{k}  : MF={mf_n_vals[k_vals.index(k)]:.4f} vs CKAN={ckan_n_vals[k_vals.index(k)]:.4f} (CKAN thay đổi: {n_diff:+.2f}%)")
print("=" * 75)

## 11. Lưu Trữ Trọng Số & Tải Checkpoint Về Máy
Lưu toàn bộ checkpoint mô hình CKAN và siêu tham số đã huấn luyện trên Book-Crossing để có thể tải về máy tính cá nhân.

In [ ]:
# ============================================================
# 11. XUẤT MÔ HÌNH VÀ TẢI VỀ MÁY TÍNH
# ============================================================
checkpoint_info = {
    "dataset": DATASET,
    "model_state_dict": ckan_model.state_dict(),
    "n_entity": n_entity,
    "n_relation": n_relation,
    "dim": DIM,
    "n_layer": N_LAYER,
    "agg": AGG,
    "test_auc": ckan_test_auc,
    "test_f1": ckan_test_f1,
    "topk_metrics": topk_cmp["CKAN"]
}

save_path = "./models/ckan_book_checkpoint.pt"
torch.save(checkpoint_info, save_path)
print(f"Đã lưu checkpoint mô hình CKAN tại: {save_path} ({os.path.getsize(save_path) / 1e6:.2f} MB)")

try:
    from google.colab import files
    print("Đang khởi tạo tải file checkpoint về máy...")
    files.download(save_path)
except Exception:
    print("Đang chạy trên môi trường cục bộ hoặc không tải được tự động qua Colab files.")